# group project
## Student performance: Spark scalability on Dataproc

Option A. I reused the cleaning and preprocessing from my [student performance project](https://github.com/timwalewangko/personal_porfolio/blob/main/Kaggle%20Analysis/student-success-factors/student-success-factors.ipynb).
The [dataset](https://www.kaggle.com/datasets/ayeshasiddiqa123/student-perfirmance) has 6,607 rows and 20 columns.

The workload cleans strings and missing values, caps scores, creates one-hot columns, standardizes numbers, joins group averages, and writes Parquet plus grouped summaries. It stops at data processing.

Cluster creation, worker counts, machine types, executor settings, and Spark properties are configured in the Google Cloud UI. This notebook uses that cluster and records its effective settings.

## 1. Connection and run

Open this notebook in the cluster's JupyterLab **PySpark kernel** through Component Gateway. Apply the fixed Spark properties from the README in the UI before starting the kernel. Replace `YOUR_BUCKET_NAME`, upload the source CSV to the path below, and keep `RUN` the same for all configurations being compared. The cluster service account needs read/write access to that bucket.

`STUDY` selects the experiment; it does not change the cluster. Use `"data"` with 2 workers, then `"workers"` for each approved worker count. After changing the cluster in the UI, wait for it to be ready and restart the kernel. Keep the UI resource settings comparable.

If available, enter the total cluster hourly estimate from the UI in `HOURLY_USD`. Leaving it as `NaN` keeps the cost estimate blank.

In [ ]:
import json
import math
import random
import socket
import time
from io import BytesIO, StringIO
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import storage
from IPython.display import display, Markdown
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F, types as T

BUCKET = "YOUR_BUCKET_NAME"
RUN = "run-01"
STUDY = "data"
HOURLY_USD = np.nan

ROOT = f"gs://{BUCKET}/comp644-project"
SOURCE = f"gs://{BUCKET}/StudentPerformanceFactors.csv"
DATA = f"{ROOT}/data"
RESULTS = f"{ROOT}/results/{RUN}"
PROCESSED = f"{ROOT}/processed/{RUN}"

client = storage.Client()
bucket = client.bucket(BUCKET)
spark = SparkSession.builder.appName("comp644-project").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
assert sc.master == "yarn"
assert spark.conf.get("spark.sql.constraintPropagation.enabled") == "false", "Set spark.sql.constraintPropagation.enabled=false in the Dataproc UI."
PARTITIONS = int(spark.conf.get("spark.sql.shuffle.partitions"))

In [ ]:
def get_json(url):
    with urlopen(Request(url, headers={"Accept": "application/json"})) as response:
        return json.load(response)

nodes = get_json("http://127.0.0.1:8088/ws/v1/cluster/nodes?states=RUNNING")["nodes"]["node"]
master = socket.gethostname().split(".")[0]
worker_nodes = [n for n in nodes if n["nodeHostName"].split(".")[0] != master]
WORKERS = len(worker_nodes)
assert WORKERS > 0
YARN_VCORES = sum(n["usedVirtualCores"] + n["availableVirtualCores"] for n in worker_nodes)
print("Workers:", WORKERS, "| YARN vcores:", YARN_VCORES, "| Partitions:", PARTITIONS)
print("Spark:", spark.version, "| Application:", sc.applicationId)

def object_name(uri):
    return uri.removeprefix(f"gs://{BUCKET}/")

def save_csv(table, uri):
    bucket.blob(object_name(uri)).upload_from_string(table.to_csv(index=False), content_type="text/csv")

def read_csv(uri):
    return pd.read_csv(StringIO(bucket.blob(object_name(uri)).download_as_text()))

def file_bytes(uri, suffix):
    return sum(b.size for b in client.list_blobs(BUCKET, prefix=object_name(uri) + "/")
               if b.name.rsplit("/", 1)[-1].startswith("part-") and b.name.endswith(suffix))

def save_figure(fig, name):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=160)
    bucket.blob(object_name(f"{RESULTS}/{name}.png")).upload_from_string(
        buffer.getvalue(), content_type="image/png")

## 2. Schema and processing

The same categories and cleaning rules are used for every run. Numeric scaling uses Spark averages and population standard deviations. Extra columns are study hours × attendance, log tutoring sessions, and previous score minus the school/resource group average.

In [ ]:
columns = [
    "hours_studied",
    "attendance",
    "parental_involvement",
    "access_to_resources",
    "extracurricular_activities",
    "sleep_hours",
    "previous_scores",
    "motivation_level",
    "internet_access",
    "tutoring_sessions",
    "family_income",
    "teacher_quality",
    "school_type",
    "peer_influence",
    "physical_activity",
    "learning_disabilities",
    "parental_education_level",
    "distance_from_home",
    "gender",
    "exam_score",
]
numeric = [
    "hours_studied",
    "attendance",
    "sleep_hours",
    "previous_scores",
    "tutoring_sessions",
    "physical_activity",
]
categorical = [c for c in columns if c not in numeric + ["exam_score"]]
schema = T.StructType(
    [
        T.StructField(
            c, T.IntegerType() if c in numeric + ["exam_score"] else T.StringType()
        )
        for c in columns
    ]
)

In [ ]:
def clean(spark, path):
    df = spark.read.option("header", True).schema(schema).csv(path)
    for c in categorical:
        df = df.withColumn(c, F.initcap(F.trim(F.col(c))))
    for c in numeric + ["exam_score"]:
        df = df.withColumn(c, F.col(c).cast("double"))
    df = df.dropna()
    return (
        df.withColumn("score_capped", (F.col("exam_score") > 100).cast("int"))
        .withColumn("exam_score", F.least(F.col("exam_score"), F.lit(100.0)))
        .withColumn(
            "study_attendance", F.col("hours_studied") * F.col("attendance") / 100
        )
        .withColumn("log_tutoring", F.log1p("tutoring_sessions"))
    )


levels = {
    "parental_involvement": ["Low", "Medium", "High"],
    "access_to_resources": ["Low", "Medium", "High"],
    "extracurricular_activities": ["No", "Yes"],
    "motivation_level": ["Low", "Medium", "High"],
    "internet_access": ["No", "Yes"],
    "family_income": ["Low", "Medium", "High"],
    "teacher_quality": ["Low", "Medium", "High"],
    "school_type": ["Private", "Public"],
    "peer_influence": ["Negative", "Neutral", "Positive"],
    "learning_disabilities": ["No", "Yes"],
    "parental_education_level": ["High School", "College", "Postgraduate"],
    "distance_from_home": ["Near", "Moderate", "Far"],
    "gender": ["Female", "Male"],
}
scaled = numeric + ["study_attendance", "log_tutoring", "previous_score_gap"]
flags = [
    (c, value, c + "_" + value.lower().replace(" ", "_"))
    for c, values in levels.items()
    for value in values
]
features = [c + "_z" for c in scaled] + [name for c, value, name in flags]


def prepare(spark, path):
    df = clean(spark, path)
    group_means = df.groupBy("school_type", "access_to_resources").agg(
        F.avg("previous_scores").alias("group_previous_score")
    )
    df = (
        df.join(F.broadcast(group_means), ["school_type", "access_to_resources"])
        .withColumn(
            "previous_score_gap",
            F.col("previous_scores") - F.col("group_previous_score"),
        )
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    stats = (
        df.agg(
            *[
                expr
                for c in scaled
                for expr in [
                    F.avg(c).alias(c + "_mean"),
                    F.stddev_pop(c).alias(c + "_std"),
                ]
            ]
        )
        .first()
        .asDict()
    )
    normalized = [
        ((F.col(c) - stats[c + "_mean"]) / stats[c + "_std"]).alias(c + "_z")
        for c in scaled
    ]
    encoded = [
        (F.col(c) == value).cast("double").alias(name) for c, value, name in flags
    ]
    return df.select("*", *normalized, *encoded), df


def materialize(spark, prepared, path):
    (
        prepared.repartition(PARTITIONS)
        .write.mode("overwrite")
        .option("compression", "snappy")
        .parquet(path)
    )
    output = spark.read.parquet(path)
    energy = sum((i + 1) * F.col(c) * F.col(c) for i, c in enumerate(features))
    return (
        output.groupBy("school_type", "access_to_resources")
        .agg(
            F.count("*").alias("rows"),
            F.sum("exam_score").alias("score_sum"),
            F.sum("score_capped").alias("capped"),
            F.sum(energy).alias("feature_sum"),
            F.avg("hours_studied").alias("mean_hours"),
        )
        .orderBy("school_type", "access_to_resources")
        .collect()
    )

## 3. Prepare the size variants — run once

Run this cell once on the 2-worker cluster. Skip it when returning with another worker count. It repeats the source rows into approximately 100 MB, 500 MB, 1 GB, and 2 GB of uncompressed CSV. Sizes below are actual object bytes, including headers; MB means 1,000,000 bytes.

Only source duplicates are removed. Intentional repetitions remain. Generation is outside the benchmark timing, and every cluster reads the same saved objects.

In [ ]:
seed = spark.read.option("header", True).schema(schema).csv(SOURCE).dropDuplicates()
seed.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{DATA}/seed")
seed_rows = seed.count()
seed_bytes = file_bytes(f"{DATA}/seed", ".csv")
manifest = []
for mb in [100, 500, 1000, 2000]:
    copies = math.ceil(mb * 1_000_000 / seed_bytes)
    files = math.ceil(mb / 100)
    expanded = spark.range(copies, numPartitions=files).crossJoin(F.broadcast(seed)).select(*columns)
    path = f"{DATA}/{mb}mb"
    expanded.write.mode("overwrite").option("header", True).csv(path)
    manifest.append(dict(size_mb=mb, actual_mb=file_bytes(path, ".csv") / 1e6,
                         rows=seed_rows * copies, copies=copies, files=files))
sizes = pd.DataFrame(manifest)
save_csv(sizes, f"{DATA}/dataset_sizes.csv")
display(sizes.round(3))

## 4. Measurement helpers

Timing includes reading CSV, cleaning, grouping/joining, normalization statistics, encoding, repartitioning, writing Snappy Parquet, and reading it for six grouped summaries. The Parquet action forces all output columns to be computed.

Generation, warm-up, UI operations, metric collection, cache removal, and saving timing tables are excluded. Each trial starts with empty Spark caches. OS and Cloud Storage caching are not reset. Throughput is input MB / total seconds.

Statistics time includes the first materialization of the joined table. Output time includes the remaining lazy transformations, Parquet output, and grouped aggregation. The Spark REST API supplies task, shuffle, CPU, and spill counters.

In [ ]:
def ui(endpoint):
    return get_json(f"{sc.uiWebUrl.rstrip('/')}/api/v1/applications/{sc.applicationId}/{endpoint}")

def completed_stages():
    return {(s["stageId"], s["attemptId"]): s for s in ui("stages?status=complete")}

def active_executors():
    return [e for e in ui("executors") if e["id"] != "driver" and e["isActive"]]

def benchmark(mb, trial):
    spark.catalog.clearCache()
    before = set(completed_stages())
    sc.setJobGroup(f"{STUDY}-w{WORKERS}-{mb}mb-t{trial}", "student performance")
    start = time.perf_counter()
    prepared, cached = prepare(spark, f"{DATA}/{mb}mb")
    ready = time.perf_counter()
    path = f"{PROCESSED}/{STUDY}/workers-{WORKERS}/{mb}mb"
    groups = materialize(spark, prepared, path)
    end = time.perf_counter()
    stages = [s for key, s in completed_stages().items() if key not in before]
    cached.unpersist(blocking=True)
    executors = active_executors()
    rows = sum(g.rows for g in groups)
    actual_mb = float(sizes.set_index("size_mb").loc[mb, "actual_mb"])
    return dict(study=STUDY, workers=WORKERS, yarn_vcores=YARN_VCORES, trial=trial,
        size_mb=mb, actual_mb=actual_mb, rows=rows, groups=len(groups), partitions=PARTITIONS,
        input_partitions=spark.read.option("header", True).schema(schema).csv(f"{DATA}/{mb}mb").rdd.getNumPartitions(),
        executor_count=len(executors), executor_cores=sum(e["totalCores"] for e in executors),
        statistics_s=ready-start, output_s=end-ready, seconds=end-start,
        mb_s=actual_mb/(end-start), mean_score=sum(g.score_sum for g in groups)/rows,
        feature_checksum=sum(g.feature_sum for g in groups)/rows, capped_rows=sum(g.capped for g in groups),
        output_mb=file_bytes(path, ".parquet")/1e6,
        tasks=sum(s["numTasks"] for s in stages),
        task_seconds=sum(s["executorRunTime"] for s in stages)/1000,
        task_cpu_seconds=sum(s["executorCpuTime"] for s in stages)/1e9,
        shuffle_read_mb=sum(s["shuffleReadBytes"] for s in stages)/1e6,
        shuffle_write_mb=sum(s["shuffleWriteBytes"] for s in stages)/1e6,
        disk_spill_mb=sum(s["diskBytesSpilled"] for s in stages)/1e6,
        hourly_usd=HOURLY_USD, estimated_cost_usd=(end-start)*HOURLY_USD/3600,
        spark_version=spark.version, application_id=sc.applicationId)

## 5. Warm-up and configuration — every cluster

Run Sections 1–2 and 4–6 again after each UI configuration change, using a fresh kernel/application. This source-sized warm-up is the same for every worker configuration. The source statistics provide an independent correctness check for all repeated datasets.

Executor count and executor cores are recorded separately from worker nodes. Save the cluster's UI configuration/export with the report to document VM types, image, disks, and pricing.

In [ ]:
sizes = read_csv(f"{DATA}/dataset_sizes.csv")
prepared, cached = prepare(spark, f"{DATA}/seed")
reference = materialize(spark, prepared, f"{PROCESSED}/{STUDY}/workers-{WORKERS}/source")
cached.unpersist(blocking=True)
source_rows = sum(g.rows for g in reference)
source_mean = sum(g.score_sum for g in reference)/source_rows
source_checksum = sum(g.feature_sum for g in reference)/source_rows
save_csv(pd.DataFrame([g.asDict() for g in reference]), f"{RESULTS}/source/{STUDY}-workers-{WORKERS}.csv")

resource_keys = ["spark.executor.instances", "spark.executor.cores", "spark.executor.memory",
                 "spark.executor.memoryOverhead", "spark.driver.memory", "spark.dynamicAllocation.enabled"]
sql_keys = ["spark.sql.shuffle.partitions", "spark.sql.files.maxPartitionBytes",
            "spark.sql.files.minPartitionNum", "spark.sql.adaptive.enabled",
            "spark.sql.constraintPropagation.enabled"]
configuration = dict(study=STUDY, workers=WORKERS, yarn_vcores=YARN_VCORES,
    application_id=sc.applicationId, spark_version=spark.version, master=sc.master,
    resources={k: sc.getConf().get(k, "default") for k in resource_keys},
    sql={k: spark.conf.get(k) for k in sql_keys},
    worker_nodes=worker_nodes,
    executors=[{k: e[k] for k in ["id", "hostPort", "totalCores", "maxMemory"]} for e in active_executors()])
bucket.blob(object_name(f"{RESULTS}/configuration/{STUDY}-workers-{WORKERS}.json")).upload_from_string(
    json.dumps(configuration, indent=2), content_type="application/json")
display(pd.DataFrame(configuration["executors"]))
print("Clean source rows:", source_rows)

## 6. Run the current experiment

**Data study:** select `STUDY = "data"` with 2 workers. Run all four sizes three times, keeping the cluster fixed.

**Worker study:** select `STUDY = "workers"`. Keep the 2 GB input fixed and repeat on each approved worker count selected in the UI. The rubric requests 1, 2, and 5 workers. Use fresh applications with the same warm-up, VM types, and per-executor settings. Only the worker/executor capacity should change.

The current configuration's CSV is saved after every trial. All configurations share `RUN` so they can be compared later. Rerunning a configuration replaces its saved trial table and processed output.

In [ ]:
if STUDY == "data":
    assert WORKERS == 2
    run_sizes = [100, 500, 1000, 2000]
else:
    run_sizes = [2000]

runs = []
for trial in [1, 2, 3]:
    for mb in random.Random(43 + trial).sample(run_sizes, len(run_sizes)):
        result = benchmark(mb, trial)
        copies = int(sizes.set_index("size_mb").loc[mb, "copies"])
        assert result["rows"] == source_rows * copies
        assert result["groups"] == 6
        np.testing.assert_allclose(result["mean_score"], source_mean, rtol=1e-9)
        np.testing.assert_allclose(result["feature_checksum"], source_checksum, rtol=1e-8)
        runs.append(result)
        save_csv(pd.DataFrame(runs), f"{RESULTS}/{STUDY}/workers-{WORKERS}.csv")
        print(f"{WORKERS} workers, {mb} MB, trial {trial}: {result['seconds']:.2f} s")
display(pd.DataFrame(runs).round(3))

## 7. Combine the cloud runs — after both studies

Run the rest after all configurations have saved results under the same `RUN`. Tables use medians; error bars show the observed minimum and maximum.

The rubric uses one worker as the speedup baseline. The code labels the smallest **measured** worker count as its baseline. If an instructor-approved configuration starts at 2 workers, efficiency is normalized to that baseline rather than presented as a one-worker result. A master-only cluster is not a one-worker baseline.

In [ ]:
raw_data = read_csv(f"{RESULTS}/data/workers-2.csv")
worker_files = [b for b in client.list_blobs(BUCKET, prefix=object_name(f"{RESULTS}/workers/"))
                if b.name.endswith(".csv")]
raw_workers = pd.concat([pd.read_csv(StringIO(b.download_as_text())) for b in worker_files], ignore_index=True)
display(raw_data[["size_mb", "trial", "rows", "input_partitions", "statistics_s", "output_s", "seconds", "mb_s"]].round(3))
display(raw_workers[["workers", "yarn_vcores", "executor_count", "executor_cores", "trial", "seconds", "mb_s"]].sort_values(["workers", "trial"]).round(3))

data_summary = raw_data.groupby("size_mb").agg(
    actual_mb=("actual_mb", "first"), rows=("rows", "first"), seconds=("seconds", "median"),
    min_s=("seconds", "min"), max_s=("seconds", "max"), statistics_s=("statistics_s", "median"),
    output_s=("output_s", "median"), shuffle_mb=("shuffle_read_mb", "median"),
    spill_mb=("disk_spill_mb", "median"), cost_usd=("estimated_cost_usd", "median"))
data_summary["mb_s"] = data_summary.actual_mb/data_summary.seconds
worker_summary = raw_workers.groupby("workers").agg(
    yarn_vcores=("yarn_vcores", "first"), executor_count=("executor_count", "median"),
    executor_cores=("executor_cores", "median"), seconds=("seconds", "median"),
    min_s=("seconds", "min"), max_s=("seconds", "max"), statistics_s=("statistics_s", "median"),
    output_s=("output_s", "median"), tasks=("tasks", "median"),
    shuffle_mb=("shuffle_read_mb", "median"), spill_mb=("disk_spill_mb", "median"),
    task_seconds=("task_seconds", "median"), cpu_seconds=("task_cpu_seconds", "median"),
    hourly_usd=("hourly_usd", "first"), cost_usd=("estimated_cost_usd", "median"))
BASELINE = int(worker_summary.index.min())
worker_summary["speedup"] = worker_summary.loc[BASELINE, "seconds"]/worker_summary.seconds
worker_summary["efficiency_pct"] = worker_summary.speedup/(worker_summary.index/BASELINE)*100
worker_summary["worker_hours"] = worker_summary.index*worker_summary.seconds/3600
worker_summary["task_cpu_share_pct"] = worker_summary.cpu_seconds/worker_summary.task_seconds*100
save_csv(data_summary.reset_index(), f"{RESULTS}/data_summary.csv")
save_csv(worker_summary.reset_index(), f"{RESULTS}/worker_summary.csv")
display(data_summary.round(3))
display(worker_summary.round(3))
print("Speedup baseline:", BASELINE, "workers")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
x = data_summary.actual_mb/1000
y = data_summary.seconds
axes[0].errorbar(x, y, yerr=[y-data_summary.min_s, data_summary.max_s-y], fmt="o-", capsize=4)
axes[0].plot(x, y.iloc[0]*x/x.iloc[0], "--", label="Proportional to first size")
axes[0].set(xlabel="Actual input (GB)", ylabel="Median seconds", title="Input size vs. time")
axes[0].legend(fontsize=8)
axes[1].plot(x, data_summary.mb_s, "o-")
axes[1].set(xlabel="Actual input (GB)", ylabel="MB/s", title="Input size vs. throughput")
fig.tight_layout()
save_figure(fig, "data_scalability")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
x = worker_summary.index
y = worker_summary.seconds
axes[0].errorbar(x, y, yerr=[y-worker_summary.min_s, worker_summary.max_s-y], fmt="o-", capsize=4)
axes[0].set(ylabel="Median seconds", title="Workers vs. time")
axes[1].plot(x, worker_summary.speedup, "o-", label="Measured")
axes[1].plot(x, x/BASELINE, "--", label="Ideal")
axes[1].set(ylabel=f"Speedup from {BASELINE} worker(s)", title="Workers vs. speedup")
axes[1].legend()
axes[2].plot(x, worker_summary.efficiency_pct, "o-")
axes[2].axhline(100, linestyle="--", color="gray")
axes[2].set(ylabel="Relative efficiency (%)", title="Workers vs. efficiency")
for ax in axes:
    ax.set_xlabel("Dataproc workers")
    ax.set_xticks(x)
fig.tight_layout()
save_figure(fig, "worker_scalability")
plt.show()

## 8. Performance answers

These answers use the cloud trial tables. Cost stays blank until an hourly estimate is provided. Worker-hours are a resource-use proxy, not the cloud invoice. Task CPU share is CPU time divided by summed task elapsed time, not cluster-wide CPU utilization.

In [ ]:
small = data_summary.iloc[0]
large = data_summary.iloc[-1]
best = int(worker_summary.seconds.idxmin())
largest = int(worker_summary.index.max())
counts = list(worker_summary.index)
previous = counts[-2]
gain = worker_summary.loc[previous, "seconds"]/worker_summary.loc[largest, "seconds"]
ideal = largest/previous
worst = data_summary.assign(seconds_per_mb=data_summary.seconds/data_summary.actual_mb).seconds_per_mb.idxmax()
display(Markdown(f"""
1. **Data growth:** Input increased {large.actual_mb/small.actual_mb:.2f}× and median runtime increased {large.seconds/small.seconds:.2f}×. Throughput changed from {small.mb_s:.2f} to {large.mb_s:.2f} MB/s. A lower runtime multiplier suggests fixed overhead was spread over more input; a higher multiplier suggests additional work such as spills or I/O pressure.

2. **More workers:** The fastest measured configuration was {best} workers. The last step, {previous}→{largest} workers, gave {gain:.2f}× speedup against an ideal {ideal:.2f}×.

3. **Diminishing returns:** At {largest} workers, speedup from the {BASELINE}-worker baseline was {worker_summary.loc[largest, 'speedup']:.2f}× and relative efficiency was {worker_summary.loc[largest, 'efficiency_pct']:.1f}%. Compare that with the preceding configuration to identify where extra capacity stopped paying off proportionally.

4. **Spark overhead:** The last configuration ran {worker_summary.loc[largest, 'tasks']:.0f} tasks, read {worker_summary.loc[largest, 'shuffle_mb']:.1f} MB of shuffle, and spilled {worker_summary.loc[largest, 'spill_mb']:.1f} MB. Its task CPU share was {worker_summary.loc[largest, 'task_cpu_share_pct']:.1f}%. Joins, repartitioning, global statistics, scheduling, and GCS output add costs beyond reading CSV. Use the UI stages and executor activity to separate possible causes.

5. **Cost:** Compare `cost_usd` in the table with runtime saved. The {largest}-worker run used {worker_summary.loc[largest, 'worker_hours']:.4f} worker-hours. The estimate covers measured processing time only; startup, idle time, storage, and network charges still need the Billing UI.

6. **Result to investigate:** The {worst} MB input had the highest seconds per MB. Its statistics/output medians were {data_summary.loc[worst, 'statistics_s']:.2f}/{data_summary.loc[worst, 'output_s']:.2f} seconds. Check its trial range and spill/shuffle changes alongside the {previous}→{largest}-worker step, which reached {gain/ideal*100:.1f}% of ideal scaling. Explain one non-obvious result using those counters and a matching Spark UI stage.
"""))

## 9. Checks and limitations

Replication increases work without adding new student information. Repeated patterns can compress and cache more easily than a larger, more diverse dataset. Results also include GCS latency, output commits, JVM warm-up, and possible differences in cloud host activity.

Three trials provide a range, not a confidence interval. Confirm the exported UI settings match across clusters and that the expected workers actually received tasks. Local measurements from the earlier version are archived under `results/local`; this notebook has no saved Dataproc measurements yet.

In [ ]:
assert raw_data.groupby("size_mb").size().eq(3).all()
assert raw_workers.groupby("workers").size().eq(3).all()
assert raw_workers.rows.nunique() == 1
assert raw_workers.groups.eq(6).all()
assert raw_workers.size_mb.eq(2000).all()
assert raw_workers.partitions.nunique() == 1
assert raw_data.application_id.nunique() == 1
assert set(raw_data.application_id).isdisjoint(set(raw_workers.application_id))
assert raw_workers.groupby("workers").application_id.nunique().eq(1).all()
assert raw_workers.application_id.nunique() == raw_workers.workers.nunique()
display(raw_workers.groupby("workers")[["mean_score", "feature_checksum"]].agg(["min", "max"]).round(6))
print("Figures and tables saved to", RESULTS)

## Sources

- Original project and dataset are linked at the top.
- [Dataproc Jupyter](https://docs.cloud.google.com/dataproc/docs/concepts/components/jupyter) and [Cloud Storage connector](https://docs.cloud.google.com/dataproc/docs/concepts/connectors/cloud-storage).
- [YARN node API](https://hadoop.apache.org/docs/stable/hadoop-yarn/hadoop-yarn-site/ResourceManagerRest.html#Cluster_Nodes_API) and [Spark monitoring](https://spark.apache.org/docs/3.5.8/monitoring.html).
- The report still needs UI screenshots, actual cloud costs, limitations, and team contributions. The presentation and optional partition bonus are separate.